In [1]:
import pandas as pd

df = pd.read_csv(
    "tabla1.csv",
    sep=',',
    encoding='utf-8-sig',  # gestisce eventuale BOM
    header=0  # usa la prima riga come intestazione
)

print(df.dtypes)

df.columns = df.columns.str.strip()

df["ANNO"] = df["ANNO"].astype(int)
df.columns = df.columns.str.strip()

df = df.sort_values(by=["ENTIDAD", "ANNO"]).reset_index(drop=True)

df["CAMBIO_PARTIDO"] = (df["VINCITORE"] != df.groupby("ENTIDAD")["VINCITORE"].shift(1)).astype(int)
df.loc[df.groupby("ENTIDAD").head(1).index, "CAMBIO_PARTIDO"] = 0
df["ANOS_EN_CARGA"] = 0
for entidad, group in df.groupby("ENTIDAD"):
    count = 0
    last_partido = None
    for idx in group.index:
        if df.loc[idx, "VINCITORE"] == last_partido:
            count += 1
        else:
            count = 1
            last_partido = df.loc[idx, "VINCITORE"]
        df.loc[idx, "ANOS_EN_CARGA"] = count
print(df.dtypes)

ANNO          int64
ENTIDAD      object
VINCITORE    object
dtype: object
ANNO               int64
ENTIDAD           object
VINCITORE         object
CAMBIO_PARTIDO     int64
ANOS_EN_CARGA      int64
dtype: object


In [21]:
inflacion_anual_estado = pd.read_csv("inflacion_anual_estado.csv", encoding="utf-8")
print(inflacion_anual_estado)


     ANNO                STATO  INFLACION_ANUAL_ESTADO
0    2000       AGUASCALIENTES                0.000000
1    2000      BAJA CALIFORNIA                0.000000
2    2000  BAJA CALIFORNIA SUR                0.000000
3    2000             CAMPECHE                0.000000
4    2000              CHIAPAS                0.000000
..    ...                  ...                     ...
827  2025           TAMAULIPAS                2.068147
828  2025             TLAXCALA                1.752988
829  2025             VERACRUZ                1.971009
830  2025              YUCATÁN                0.950675
831  2025            ZACATECAS                2.397889

[832 rows x 3 columns]


In [22]:
inflacion_anual_estado = inflacion_anual_estado.rename(columns={"STATO":"ENTIDAD","INFLACION_ANUAL":"INFLACION"})

df["ANNO"] = df["ANNO"].astype(int)
df["ENTIDAD"] = df["ENTIDAD"].astype(str)
inflacion_anual_estado["ENTIDAD"] = inflacion_anual_estado["ENTIDAD"].astype(str)

df["ENTIDAD"] = df["ENTIDAD"].str.strip().str.upper()
inflacion_anual_estado["ENTIDAD"] = inflacion_anual_estado["ENTIDAD"].str.strip().str.upper()




In [25]:
merged = df.merge(
    inflacion_anual_estado.rename(columns={"STATO":"ENTIDAD","INFLACION_ANUAL":"INFLACION"}),
    how="outer",
    on=["ANNO","ENTIDAD"],
    indicator=True
)

merged=merged.drop(columns=["_merge"])
merged.to_csv("governadores_con_inflacion.csv", index=False)
print(merged.dtypes)


ANNO                        int64
ENTIDAD                    object
VINCITORE                  object
CAMBIO_PARTIDO            float64
ANOS_EN_CARGA             float64
INFLACION_ANUAL_ESTADO    float64
dtype: object
